# Assignment 6: CNN for tomato leaf disease classification
Uses the PlantVillage dataset (via TensorFlow Datasets) filtered to the tomato classes (10 classes: 9 diseases + healthy).

In [ ]:
!pip install -q tensorflow-datasets
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf, tensorflow_datasets as tfds
from tensorflow import keras
from sklearn.metrics import confusion_matrix, classification_report
tf.keras.utils.set_random_seed(42)
IMG, BATCH = 128, 32

## 1. Load and filter to tomato classes

In [ ]:
ds, info = tfds.load("plant_village", split="train", as_supervised=True, with_info=True)
all_names = info.features["label"].names
tomato_ids = [i for i, n in enumerate(all_names) if n.lower().startswith("tomato")]
class_names = [all_names[i].replace("Tomato___", "") for i in tomato_ids]
print(len(class_names), "classes:", class_names)

mapping = np.full(len(all_names), -1, dtype=np.int64)
for new, old in enumerate(tomato_ids): mapping[old] = new
mapping = tf.constant(mapping)

ds = ds.filter(lambda x, y: mapping[y] >= 0).map(lambda x, y: (tf.image.resize(x, (IMG, IMG)), mapping[y]), num_parallel_calls=tf.data.AUTOTUNE)
n_total = ds.reduce(0, lambda c, _: c + 1).numpy()
print("Tomato images:", n_total)

## 2. Train / validation / test split (70/15/15)

In [ ]:
ds = ds.shuffle(5000, seed=42, reshuffle_each_iteration=False)
n_train, n_val = int(0.7 * n_total), int(0.15 * n_total)
train_ds = ds.take(n_train)
val_ds = ds.skip(n_train).take(n_val)
test_ds = ds.skip(n_train + n_val)

def prep(d, shuffle=False):
    if shuffle: d = d.shuffle(2000)
    return d.batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_b, val_b, test_b = prep(train_ds, True), prep(val_ds), prep(test_ds)

# preview
plt.figure(figsize=(12, 5))
for x, y in train_b.take(1):
    for i in range(10):
        plt.subplot(2, 5, i+1); plt.imshow(x[i].numpy().astype("uint8")); plt.title(class_names[int(y[i])], fontsize=7); plt.axis("off")
plt.show()

## 3. CNN model
Augmentation + normalization inside the model, 3 conv blocks (Conv-BN-Pool), dropout, softmax head.

In [ ]:
aug = keras.Sequential([keras.layers.RandomFlip("horizontal_and_vertical"), keras.layers.RandomRotation(0.1), keras.layers.RandomZoom(0.1)])

def conv_block(x, f):
    x = keras.layers.Conv2D(f, 3, padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    return keras.layers.MaxPooling2D()(x)

inp = keras.Input((IMG, IMG, 3))
x = aug(inp)
x = keras.layers.Rescaling(1/255.)(x)
for f in [32, 64, 128]: x = conv_block(x, f)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dropout(0.4)(x)
out = keras.layers.Dense(len(class_names), activation="softmax")(x)
model = keras.Model(inp, out)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
cb = [keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True), keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5)]
hist = model.fit(train_b, validation_data=val_b, epochs=15, callbacks=cb)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="val"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.show()

## 4. Evaluation

In [ ]:
loss, acc = model.evaluate(test_b, verbose=0)
print(f"Test accuracy: {acc:.4f}")
y_true = np.concatenate([y.numpy() for _, y in test_b])
y_pred = np.argmax(model.predict(test_b, verbose=0), axis=1)
print(classification_report(y_true, y_pred, target_names=class_names))
plt.figure(figsize=(9, 7))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.xticks(rotation=60, ha="right"); plt.show()

model.save("tomato_cnn.keras")

## Conclusion
A compact CNN with augmentation classifies tomato leaf diseases with high accuracy on PlantVillage. Note: PlantVillage images are lab-style (plain background); real field photos will give lower accuracy.